
# MUBIO07 · Programación en Python
## Actividad grupal 3 · Inteligencia artificial con Python

### Equipo
- **Meritxell Bastardas Pernil**
- **Rubén Juárez Cádiz**
- **Águeda Sobrino Martínez**
- **Yiling Teng Fang**

### Objetivos
1. Documentar la creación y compartición de un cuaderno colaborativo en **Google Colab**.
2. Implementar y comparar **SVM, Random Forest y Naive Bayes** sobre el dataset `iris` de Seaborn.
3. Diseñar, entrenar y evaluar una **red neuronal convolucional (CNN)** para clasificar células **Parasitized / Uninfected** del dataset de malaria.

> **Reproducibilidad:** se fija una semilla global (`SEED = 42`) y se documentan las decisiones de partición, preprocesamiento, métricas y arquitectura.
>
> **Uso educativo:** el clasificador de malaria se desarrolla como ejercicio académico y no constituye un sistema de diagnóstico clínico.



# 1. Creación y uso colaborativo de Google Colab — 2 puntos

## 1A. Creación del documento
El procedimiento seguido para crear el cuaderno es:

1. Acceder a **Google Colab** con una cuenta de Google.
2. Seleccionar **Archivo → Nuevo cuaderno**.
3. Renombrar el archivo como `MUBIO07_Actividad3_IA_Python_Grupo.ipynb`.
4. Guardarlo en una carpeta compartida de Google Drive del equipo.
5. En **Entorno de ejecución → Cambiar tipo de entorno de ejecución**, seleccionar **GPU** para la fase de entrenamiento de la CNN cuando esté disponible.

## 1B. Compartición con el equipo
Para trabajar de forma colaborativa:

1. Pulsar **Compartir** en la esquina superior derecha.
2. Mantener el acceso general como **Restringido**.
3. Añadir a los cuatro integrantes del equipo mediante sus cuentas de Google:
   - Meritxell Bastardas Pernil
   - Rubén Juárez Cádiz
   - Águeda Sobrino Martínez
   - Yiling Teng Fang
4. Asignar a todos el rol **Editor**.
5. Enviar la invitación.
6. Comprobar desde **Compartir → Personas con acceso** que los cuatro integrantes figuran con permisos de edición.
7. Utilizar el historial de versiones de Drive/Colab para conservar la trazabilidad de los cambios.

Las direcciones de correo no se publican en GitHub para evitar exponer datos personales. La comprobación de acceso se realiza desde la interfaz de Google Drive/Colab.



# 2. Machine learning con el dataset Iris — 4,5 puntos

Se utilizará exactamente el dataset solicitado en el enunciado:

```python
iris = sns.load_dataset("iris")
```

Las clases se vectorizan primero mediante **One Hot Encoding**. Para entrenar los clasificadores de Scikit-learn se usa después el índice de la clase (`argmax`), porque `SVC` y `GaussianNB` esperan una variable objetivo unidimensional para un problema multiclase estándar.

Se emplea una partición **80 % entrenamiento / 20 % prueba**, estratificada por especie. Con 150 muestras, esto deja 120 para entrenamiento y 30 para prueba, manteniendo la misma proporción de las tres clases.


In [ ]:

import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore")

iris = sns.load_dataset("iris")
print(f"Dimensiones de Iris: {iris.shape}")
display(iris.head())
print("\nDistribución de clases:")
display(iris["species"].value_counts())


In [ ]:

# 2A. One Hot Encoding de la variable de clase
X_iris = iris.drop(columns="species").copy()
y_texto = iris["species"].to_numpy()

try:
    one_hot = OneHotEncoder(sparse_output=False)
except TypeError:  # compatibilidad con versiones antiguas de scikit-learn
    one_hot = OneHotEncoder(sparse=False)

y_onehot = one_hot.fit_transform(y_texto.reshape(-1, 1))
clases_iris = one_hot.categories_[0]

df_onehot = pd.DataFrame(
    y_onehot,
    columns=[f"species_{c}" for c in clases_iris],
    index=iris.index,
)

print("Clases codificadas:", list(clases_iris))
display(pd.concat([iris[["species"]], df_onehot], axis=1).head(12))

# Índice entero de clase para los clasificadores de scikit-learn.
y_iris = np.argmax(y_onehot, axis=1)


In [ ]:

# 2B. División 80/20, estratificada y reproducible.
indices = np.arange(len(iris))

idx_train, idx_test = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
    stratify=y_iris,
)

X_train = X_iris.iloc[idx_train].copy()
X_test = X_iris.iloc[idx_test].copy()
y_train = y_iris[idx_train]
y_test = y_iris[idx_test]

print(f"Entrenamiento: {len(X_train)} muestras ({len(X_train)/len(iris):.0%})")
print(f"Prueba:         {len(X_test)} muestras ({len(X_test)/len(iris):.0%})")

print("\nDistribución en entrenamiento:", np.bincount(y_train))
print("Distribución en prueba:       ", np.bincount(y_test))


In [ ]:

# 2C. SVM, Random Forest y Naive Bayes
#
# SVM y GaussianNB se incluyen en pipelines con StandardScaler.
# Random Forest no necesita escalado porque se basa en particiones del espacio
# de características mediante árboles.

modelos = {
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            random_state=SEED,
        )),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1,
    ),
    "Naive Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("model", GaussianNB()),
    ]),
}

resultados = []
predicciones = {}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    predicciones[nombre] = y_pred

    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision_macro": precision_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "Recall_macro": recall_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "F1_macro": f1_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
    })

resultados_df = (
    pd.DataFrame(resultados)
    .sort_values(["F1_macro", "Accuracy"], ascending=False)
    .reset_index(drop=True)
)

display(
    resultados_df.style.format({
        "Accuracy": "{:.3f}",
        "Precision_macro": "{:.3f}",
        "Recall_macro": "{:.3f}",
        "F1_macro": "{:.3f}",
    })
)

for nombre, y_pred in predicciones.items():
    print("\n" + "=" * 70)
    print(nombre)
    print("=" * 70)
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=clases_iris,
            digits=3,
            zero_division=0,
        )
    )


In [ ]:

# Matrices de confusión de los tres modelos
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (nombre, y_pred) in zip(axes, predicciones.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=clases_iris,
        yticklabels=clases_iris,
        ax=ax,
    )
    ax.set_title(nombre)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Clase real")

plt.tight_layout()
plt.show()



## 2D. Selección del mejor modelo

Para evitar escoger un modelo únicamente por **accuracy**, el criterio principal será el **F1 macro**, porque pondera por igual el rendimiento de las tres especies y combina precisión y recall.

Como análisis adicional de robustez se realiza una **validación cruzada estratificada de 5 particiones**. Esta segunda evidencia es especialmente útil si dos modelos empatan o quedan muy próximos en el conjunto de prueba.


In [ ]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
}

cv_rows = []

for nombre, modelo in modelos.items():
    scores = cross_validate(
        modelo,
        X_iris,
        y_iris,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
    )
    cv_rows.append({
        "Modelo": nombre,
        "CV_accuracy_media": scores["test_accuracy"].mean(),
        "CV_accuracy_std": scores["test_accuracy"].std(),
        "CV_F1_macro_media": scores["test_f1_macro"].mean(),
        "CV_F1_macro_std": scores["test_f1_macro"].std(),
    })

cv_df = (
    pd.DataFrame(cv_rows)
    .sort_values("CV_F1_macro_media", ascending=False)
    .reset_index(drop=True)
)

display(
    cv_df.style.format({
        "CV_accuracy_media": "{:.3f}",
        "CV_accuracy_std": "{:.3f}",
        "CV_F1_macro_media": "{:.3f}",
        "CV_F1_macro_std": "{:.3f}",
    })
)

max_f1 = resultados_df["F1_macro"].max()
ganadores_test = resultados_df[
    np.isclose(resultados_df["F1_macro"], max_f1, atol=1e-12)
]["Modelo"].tolist()

if len(ganadores_test) == 1:
    conclusion = (
        f"En el hold-out 80/20, **{ganadores_test[0]}** obtiene el mayor F1 macro "
        f"({max_f1:.3f}). La validación cruzada se utiliza para comprobar que "
        "la conclusión no depende únicamente de una partición."
    )
else:
    mejor_cv = cv_df.iloc[0]["Modelo"]
    conclusion = (
        f"En el hold-out existe un empate en F1 macro entre "
        f"**{', '.join(ganadores_test)}** ({max_f1:.3f}). "
        f"Como criterio de desempate se considera la validación cruzada; "
        f"el mayor F1 macro medio corresponde a **{mejor_cv}** "
        f"({cv_df.iloc[0]['CV_F1_macro_media']:.3f})."
    )

display(Markdown("### Conclusión automática\n" + conclusion))



### Interpretación técnica de modelos con rendimiento similar

Si las métricas resultan muy próximas:

- **SVM (RBF)** suele ser una opción sólida cuando existen fronteras no lineales entre clases y el número de muestras es moderado. El escalado es importante porque trabaja con distancias.
- **Random Forest** es robusto, no necesita escalado y permite inspeccionar importancia de variables, pero en datasets pequeños puede presentar cierta variabilidad entre particiones.
- **Gaussian Naive Bayes** es extremadamente rápido y funciona bien cuando la aproximación de independencia condicional y distribución gaussiana de las variables es razonable. Su simplicidad puede ser una ventaja cuando se busca un modelo ligero.

La elección final se fundamenta en las métricas calculadas en la ejecución y no en una preferencia previa por un algoritmo.



# 3. Deep learning: clasificación de células de malaria — 3,5 puntos

El dataset proporcionado contiene dos clases:

- `Parasitized`: célula parasitada.
- `Uninfected`: célula no infectada.

## Decisiones de diseño

- **Tamaño de entrada:** `100 × 100` píxeles, como propone el enunciado.
- **Canales:** **3 (RGB)**. Las imágenes originales son en color y la tinción aporta información cromática potencialmente discriminativa; convertirlas a escala de grises descartaría esa señal.
- **Normalización:** valores de píxel a `[0, 1]`.
- **Salida:** una neurona con activación **sigmoid**, porque es una clasificación binaria.
- **Pérdida:** `binary_crossentropy`, adecuada para una variable objetivo binaria y una salida probabilística sigmoid.
- **Split:** aproximadamente 70 % entrenamiento, 15 % validación y 15 % prueba.
- **Control de fuga de información:** la partición se hace por **identificador de paciente extraído del nombre del fichero**, de modo que imágenes de un mismo paciente no aparecen simultáneamente en entrenamiento y evaluación.


In [ ]:

# Preparación de rutas para Google Colab / ejecución local.
from pathlib import Path
import os
import re
import sys
import zipfile

# En Colab se puede montar Drive para mantener el dataset fuera del repositorio.
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

candidatos_zip = [
    Path("/content/drive/MyDrive/MUBIO07/Malaria Data.zip"),
    Path("/content/Malaria Data.zip"),
    Path("data/Malaria Data.zip"),
]

MALARIA_ZIP = next((p for p in candidatos_zip if p.exists()), None)

if MALARIA_ZIP is None:
    raise FileNotFoundError(
        "No se ha encontrado 'Malaria Data.zip'. "
        "Súbelo a /content, colócalo en data/, o guárdalo en "
        "/content/drive/MyDrive/MUBIO07/."
    )

if "google.colab" in sys.modules:
    EXTRACT_ROOT = Path("/content/mubio07_malaria")
else:
    EXTRACT_ROOT = Path("data/extracted")

DATASET_DIR = EXTRACT_ROOT / "Malaria"

if not (DATASET_DIR / "Parasitized").exists():
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Extrayendo {MALARIA_ZIP} ...")
    with zipfile.ZipFile(MALARIA_ZIP, "r") as zf:
        zf.extractall(EXTRACT_ROOT)

assert (DATASET_DIR / "Parasitized").is_dir()
assert (DATASET_DIR / "Uninfected").is_dir()

print("Dataset:", DATASET_DIR.resolve())


In [ ]:

# 3A. Auditoría del dataset y creación de metadatos
from sklearn.model_selection import StratifiedGroupKFold

def extraer_patient_id(path):
    """
    Extrae el prefijo anterior a 'thin' del nombre del fichero.
    Ejemplos:
      C100P61ThinF_...  -> C100P61
      C37BP2_thinF_...  -> C37BP2
    """
    nombre = Path(path).name
    return re.split(r"(?i)thin", nombre, maxsplit=1)[0].rstrip("_").upper()

filas = []
for clase, label in [("Uninfected", 0), ("Parasitized", 1)]:
    for ruta in sorted((DATASET_DIR / clase).glob("*.png")):
        filas.append({
            "path": str(ruta),
            "class_name": clase,
            "label": label,
            "patient_id": extraer_patient_id(ruta),
        })

metadata = pd.DataFrame(filas)

print(f"Total de imágenes: {len(metadata):,}")
display(metadata["class_name"].value_counts().rename("n").to_frame())
print(f"Pacientes/grupos identificados: {metadata['patient_id'].nunique()}")

assert metadata["label"].nunique() == 2
assert metadata["path"].is_unique

# Split estratificado y agrupado:
# 20 folds -> 14 train (70%), 3 validation (15%), 3 test (15%).
sgkf = StratifiedGroupKFold(
    n_splits=20,
    shuffle=True,
    random_state=SEED,
)

metadata["fold"] = -1

for fold_id, (_, idx_fold) in enumerate(
    sgkf.split(
        metadata,
        y=metadata["label"],
        groups=metadata["patient_id"],
    )
):
    metadata.loc[idx_fold, "fold"] = fold_id

train_df = metadata[metadata["fold"] < 14].reset_index(drop=True)
val_df = metadata[
    (metadata["fold"] >= 14) & (metadata["fold"] < 17)
].reset_index(drop=True)
test_df = metadata[metadata["fold"] >= 17].reset_index(drop=True)

def resumen_split(nombre, df):
    return {
        "Split": nombre,
        "Imágenes": len(df),
        "Porcentaje": len(df) / len(metadata),
        "Pacientes": df["patient_id"].nunique(),
        "% Parasitized": df["label"].mean(),
    }

split_df = pd.DataFrame([
    resumen_split("Train", train_df),
    resumen_split("Validation", val_df),
    resumen_split("Test", test_df),
])

display(
    split_df.style.format({
        "Porcentaje": "{:.1%}",
        "% Parasitized": "{:.1%}",
    })
)

# Comprobación explícita de ausencia de fuga por paciente.
assert set(train_df.patient_id).isdisjoint(set(val_df.patient_id))
assert set(train_df.patient_id).isdisjoint(set(test_df.patient_id))
assert set(val_df.patient_id).isdisjoint(set(test_df.patient_id))

print("✓ No hay patient_id compartidos entre train, validation y test.")


In [ ]:

# Preprocesamiento con OpenCV: RGB, 100x100 y normalización [0, 1].
import cv2

IMG_SIZE = (100, 100)
N_CHANNELS = 3

def cargar_imagen_opencv(path, img_size=IMG_SIZE):
    imagen = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if imagen is None:
        raise ValueError(f"No se pudo leer la imagen: {path}")

    # OpenCV carga BGR; se convierte a RGB para visualización/modelado coherente.
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    imagen = cv2.resize(
        imagen,
        img_size,
        interpolation=cv2.INTER_AREA,
    )
    imagen = imagen.astype(np.float32) / 255.0
    return imagen

# Verificación del preprocesamiento.
ejemplo = cargar_imagen_opencv(train_df.iloc[0]["path"])
print("Shape:", ejemplo.shape)
print("dtype:", ejemplo.dtype)
print("Rango:", float(ejemplo.min()), "→", float(ejemplo.max()))

assert ejemplo.shape == (100, 100, 3)
assert 0.0 <= ejemplo.min() <= ejemplo.max() <= 1.0


In [ ]:

# Visualización de ejemplos de ambas clases.
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for fila, clase in enumerate(["Uninfected", "Parasitized"]):
    ejemplos = (
        train_df[train_df["class_name"] == clase]
        .sample(4, random_state=SEED)
        .reset_index(drop=True)
    )

    for col, row in ejemplos.iterrows():
        img = cargar_imagen_opencv(row["path"])
        axes[fila, col].imshow(img)
        axes[fila, col].set_title(clase)
        axes[fila, col].axis("off")

plt.suptitle("Ejemplos preprocesados (100×100 RGB)")
plt.tight_layout()
plt.show()



## 3B. Diseño de la CNN

La arquitectura combina:

1. **Data augmentation** moderado durante entrenamiento.
2. Cuatro bloques convolucionales con distinto número de filtros y kernels `3×3` / `5×5`.
3. `BatchNormalization` para estabilizar el entrenamiento.
4. `MaxPooling2D` para reducir progresivamente la dimensionalidad.
5. `Dropout` para reducir sobreajuste.
6. `GlobalAveragePooling2D` para reducir parámetros antes de la parte densa.
7. Una capa `Dense(1, activation="sigmoid")` para la clasificación binaria.

Las imágenes se cargan por lotes mediante una `Sequence` que utiliza **OpenCV**, evitando mantener las 27.558 imágenes simultáneamente en memoria.


In [ ]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(SEED)

BATCH_SIZE = 64

class MalariaSequence(keras.utils.Sequence):
    def __init__(
        self,
        dataframe,
        batch_size=BATCH_SIZE,
        img_size=IMG_SIZE,
        shuffle=False,
        seed=SEED,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.df = dataframe.reset_index(drop=True).copy()
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle
        self.rng = np.random.default_rng(seed)
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, batch_index):
        inicio = batch_index * self.batch_size
        fin = min(inicio + self.batch_size, len(self.df))
        idx = self.indices[inicio:fin]
        batch = self.df.iloc[idx]

        X = np.stack([
            cargar_imagen_opencv(path, self.img_size)
            for path in batch["path"]
        ]).astype(np.float32)

        y = batch["label"].to_numpy(dtype=np.float32).reshape(-1, 1)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.indices)

train_seq = MalariaSequence(train_df, shuffle=True)
val_seq = MalariaSequence(val_df, shuffle=False)
test_seq = MalariaSequence(test_df, shuffle=False)

# Pesos de clase calculados solo con entrenamiento.
classes = np.array([0, 1])
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].to_numpy(),
)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print("Class weights:", class_weight)


In [ ]:

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal_and_vertical", seed=SEED),
        layers.RandomRotation(0.08, seed=SEED),
        layers.RandomZoom(0.10, seed=SEED),
    ],
    name="data_augmentation",
)

inputs = keras.Input(shape=(100, 100, 3), name="imagen_rgb")
x = data_augmentation(inputs)

# Bloque 1
x = layers.Conv2D(32, (3, 3), padding="same", use_bias=False)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling2D((2, 2))(x)

# Bloque 2: kernel distinto (5x5)
x = layers.Conv2D(64, (5, 5), padding="same", use_bias=False)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Dropout(0.15)(x)

# Bloque 3
x = layers.Conv2D(128, (3, 3), padding="same", use_bias=False)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Dropout(0.25)(x)

# Bloque 4
x = layers.Conv2D(256, (3, 3), padding="same", use_bias=False)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.45)(x)

# Clasificación binaria -> una probabilidad P(Parasitized)
outputs = layers.Dense(1, activation="sigmoid", name="prob_parasitized")(x)

model = keras.Model(inputs, outputs, name="malaria_cnn")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc"),
    ],
)

model.summary()



### Justificación de la última capa y la pérdida

- **Activación final: `sigmoid`.** Devuelve un valor entre 0 y 1 interpretable como probabilidad de pertenecer a la clase `Parasitized`.
- **Pérdida: `binary_crossentropy`.** Es la función de pérdida estándar para clasificación binaria con una salida sigmoid.
- **Tres canales RGB.** Se conservan los tres canales originales porque la información cromática de la tinción puede contribuir a distinguir estructuras celulares y parasitarias. La conversión a escala de grises eliminaría esa información.


In [ ]:

# Entrenamiento con mecanismos de control de sobreajuste.
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        "outputs/malaria_cnn_best.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
]

history = model.fit(
    train_seq,
    validation_data=val_seq,
    epochs=20,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)



# 3C. Evolución del entrenamiento y evaluación final

Se representan en una única figura:

- pérdida de entrenamiento y validación;
- accuracy de entrenamiento y validación.

Después se evalúa **una sola vez** el conjunto de prueba, que no se ha utilizado para ajustar pesos ni hiperparámetros.


In [ ]:

# Curvas de aprendizaje
hist = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(hist.index + 1, hist["loss"], label="Train")
axes[0].plot(hist.index + 1, hist["val_loss"], label="Validation")
axes[0].set_title("Evolución de la pérdida")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Binary cross-entropy")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(hist.index + 1, hist["accuracy"], label="Train")
axes[1].plot(hist.index + 1, hist["val_accuracy"], label="Validation")
axes[1].set_title("Evolución de la accuracy")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:

# Evaluación sobre TEST
test_metrics = model.evaluate(test_seq, return_dict=True, verbose=1)
display(pd.DataFrame([test_metrics]).T.rename(columns={0: "Valor"}))

y_prob = model.predict(test_seq, verbose=1).ravel()
y_pred = (y_prob >= 0.5).astype(int)
y_true = test_df["label"].to_numpy()

print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Uninfected", "Parasitized"],
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Uninfected", "Parasitized"],
    yticklabels=["Uninfected", "Parasitized"],
)
plt.title("Matriz de confusión — conjunto de prueba")
plt.xlabel("Predicción")
plt.ylabel("Clase real")
plt.tight_layout()
plt.show()


In [ ]:

# Ejemplos de predicciones correctas e incorrectas.
correctos = np.flatnonzero(y_pred == y_true)
incorrectos = np.flatnonzero(y_pred != y_true)

rng = np.random.default_rng(SEED)

n_correct = min(4, len(correctos))
n_wrong = min(4, len(incorrectos))

sel_correct = (
    rng.choice(correctos, size=n_correct, replace=False)
    if n_correct else np.array([], dtype=int)
)
sel_wrong = (
    rng.choice(incorrectos, size=n_wrong, replace=False)
    if n_wrong else np.array([], dtype=int)
)

seleccion = list(sel_correct) + list(sel_wrong)
tipos = ["Correcta"] * n_correct + ["Incorrecta"] * n_wrong

if seleccion:
    fig, axes = plt.subplots(
        2,
        4,
        figsize=(14, 7),
        squeeze=False,
    )

    for ax in axes.ravel():
        ax.axis("off")

    for pos, (idx, tipo) in enumerate(zip(seleccion, tipos)):
        ax = axes.ravel()[pos]
        row = test_df.iloc[idx]
        img = cargar_imagen_opencv(row["path"])
        real = row["class_name"]
        pred = "Parasitized" if y_pred[idx] == 1 else "Uninfected"
        prob = y_prob[idx]

        ax.imshow(img)
        ax.set_title(
            f"{tipo}\nReal: {real}\nPred: {pred} · p={prob:.3f}",
            fontsize=9,
        )
        ax.axis("off")

    plt.suptitle("Ejemplos de aciertos y errores del modelo")
    plt.tight_layout()
    plt.show()
else:
    print("No hay ejemplos disponibles para mostrar.")


In [ ]:

# Resumen final reproducible de resultados.
cnn_summary = {
    "n_total": int(len(metadata)),
    "n_train": int(len(train_df)),
    "n_validation": int(len(val_df)),
    "n_test": int(len(test_df)),
    "test_accuracy": float(accuracy_score(y_true, y_pred)),
    "test_precision": float(
        precision_score(y_true, y_pred, zero_division=0)
    ),
    "test_recall": float(
        recall_score(y_true, y_pred, zero_division=0)
    ),
    "test_f1": float(
        f1_score(y_true, y_pred, zero_division=0)
    ),
}

resumen_final = pd.DataFrame([cnn_summary])
display(resumen_final)

resumen_final.to_csv(
    "outputs/metricas_finales_malaria.csv",
    index=False,
)
resultados_df.to_csv(
    "outputs/metricas_modelos_iris.csv",
    index=False,
)

display(Markdown(
    "## Conclusión final\n"
    f"La CNN alcanza en el conjunto de prueba una **accuracy de "
    f"{cnn_summary['test_accuracy']:.3f}**, un **recall para la clase "
    f"parasitada de {cnn_summary['test_recall']:.3f}** y un **F1 de "
    f"{cnn_summary['test_f1']:.3f}**. "
    "La interpretación debe realizarse conjuntamente con la matriz de "
    "confusión y los ejemplos de error, no solo con la accuracy."
))



# Consideraciones finales de calidad

1. **No se inventan métricas:** los valores finales se generan al ejecutar el cuaderno.
2. **Reproducibilidad:** todas las particiones y operaciones pseudoaleatorias usan una semilla fija.
3. **Ausencia de fuga por paciente:** los identificadores de paciente son disjuntos entre entrenamiento, validación y prueba.
4. **Preprocesamiento verificable:** OpenCV transforma las imágenes a `100×100×3`, RGB y `[0,1]`.
5. **Evaluación completa:** curvas de aprendizaje, classification report, matriz de confusión y ejemplos de errores.
6. **Uso adecuado del test:** el conjunto de prueba se reserva para la evaluación final.
7. **Trazabilidad grupal:** el repositorio identifica a los cuatro integrantes sin publicar correos electrónicos ni otros datos de contacto.
